<a href="https://colab.research.google.com/github/JeswanthReddy78/DSCI-2025-TEAM-F/blob/main/Enriched_county_feature_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP 1: Import Libraries and Load Files


In [1]:
# STEP 1: Import necessary libraries
import pandas as pd
import numpy as np

# Load the core data files
county_df = pd.read_csv("DATA002_county_feature_dataframe_v1.csv")
superfund_df = pd.read_csv("DATA001_superfund_feature_dataframe_v1.csv")
superfund_node_map = pd.read_csv("DATA001_superfund_node_id_map_v1.csv")

# Preview first few rows
county_df.head(), superfund_df.head(), superfund_node_map.head()


(            county         state   fips  cancer_rate  avg_annual_count  \
 0  traverse county     minnesota  27155        693.5                37   
 1      polk county         texas  48373        679.5               436   
 2       galax city      virginia  51640        655.0                55   
 3   greeley county      nebraska  31077        653.1                21   
 4     dewey county  south dakota  46041        634.5                28   
 
    five_year_trend  has_cancer_data  rural_rural  rural_urban  trend_  \
 0              2.2                1         True        False   False   
 1             -0.3                1         True        False   False   
 2              1.7                1         True        False   False   
 3              0.7                1         True        False   False   
 4              1.3                1         True        False   False   
 
    trend_falling  trend_rising  trend_stable  
 0          False         False          True  
 1    

Step 2: Classify Superfund Sites by Risk and Listing Age
We will use two columns from the Superfund dataset:

hazard_rank_score – to assess the toxicity level of each site

final_list_days – to determine how recently the site was listed

We categorize these values as follows:

Risk (based on hazard_rank_score):

Low: score <= 30

Medium: 30 < score <= 50

High: score > 50

Age (based on final_list_days):

Old: days < 13000

Medium: 13000 <= days < 15000

Recent: days >= 15000

We will also remove rows with invalid final_list_days (i.e., equal to -1).

In [2]:
# Remove rows where final_list_days is invalid (-1)
superfund_df = superfund_df[superfund_df['final_list_days'] != -1].copy()

# Define a function to classify each site based on score and days
def classify_edge(score, days):
    # Risk classification
    if score > 50:
        risk = "High"
    elif score > 30:
        risk = "Medium"
    else:
        risk = "Low"

    # Age classification
    if days >= 15000:
        age = "Recent"
    elif days >= 13000:
        age = "Medium"
    else:
        age = "Old"

    return f"{risk}_{age}"

# Apply the classification function
superfund_df["edge_tag"] = superfund_df.apply(
    lambda row: classify_edge(row["hazard_rank_score"], row["final_list_days"]),
    axis=1
)

# Display the distribution of edge tags
superfund_df["edge_tag"].value_counts()


,count
edge_tag,
Medium_Old,622
Medium_Medium,366
Medium_Recent,232
High_Old,110
High_Recent,96
High_Medium,54
Low_Old,39
Low_Medium,38
Low_Recent,18


In this step, we classified each Superfund site into one of nine categories (e.g., High_Recent, Medium_Old) based on two factors:

Risk Level: Determined by the hazard score (hazard_rank_score). We used thresholds to classify sites as Low, Medium, or High risk.

Listing Age: Determined by how long ago the site was listed (final_list_days). We grouped sites into Old, Medium, or Recent based on the number of days since 1970.

This classification helps us identify what type of environmental risk each site poses and when it was added to the priority list. These tags will later be used to create node features for counties based on the types of Superfund sites they are connected

Step 3: Link Edge Tags to Counties Using FIPS
In this step, we will associate each Superfund site's edge_tag (like High_Old) with the corresponding county using the fips field from the Superfund node ID map. This step is important because we want to know how many sites of each tag type are connected to each county.

In [3]:
# Step 3: Merge edge_tag from superfund_df with fips from superfund_node_map
# We only need 'site_id' and 'edge_tag' from the superfund data
superfund_tagged = superfund_df[['site_id', 'edge_tag']]

# Merge with node map to get fips for each site
superfund_with_fips = pd.merge(superfund_node_map, superfund_tagged, on='site_id')

# Preview the result
superfund_with_fips.head()


,row_index,site_id,fips,edge_tag
0,0,100041,9150,Medium_Old
1,1,100108,9130,Medium_Old
2,2,100121,9150,Medium_Medium
3,3,100124,9110,Medium_Recent
4,4,100125,9150,Medium_Recent


Each row represents a different Superfund site, not a county.

So, if multiple Superfund sites in the same county (same FIPS) have the same tag (e.g., Medium_Recent), they will appear in multiple rows.

Count Edge Tags Per County

In this step, we will calculate how many Superfund sites of each edge_tag type are connected to each county (fips). This gives us 9 new features for each county — one column for each of the edge tag types (like High_Recent, Medium_Old, etc.).

In [4]:
# Step 4: Count the number of edge_tag types per county
# Group by fips and edge_tag, then count
edge_counts = superfund_with_fips.groupby(['fips', 'edge_tag']).size().reset_index(name='count')

# Pivot the table so that each edge_tag becomes a column
edge_tag_matrix = edge_counts.pivot(index='fips', columns='edge_tag', values='count')

# Fill missing values with 0 and convert to integer type
edge_tag_matrix = edge_tag_matrix.fillna(0).astype(int).reset_index()

# Preview the result
edge_tag_matrix.head()


edge_tag,fips,High_Medium,High_Old,High_Recent,Low_Medium,Low_Old,Low_Recent,Medium_Medium,Medium_Old,Medium_Recent
0,1003,0,0,0,0,0,0,0,0,1
1,1013,0,0,1,0,0,0,0,0,0
2,1015,1,0,0,0,0,0,0,0,0
3,1067,0,1,0,0,0,0,0,0,0
4,1073,0,0,0,0,0,0,1,0,0


Each row corresponds to a county (fips), and each column corresponds to an edge tag, such as High_Old, Medium_Recent, Low_Old, etc.

The values represent the number of Superfund sites of each type linked to that county.

Merge Edge Tag Counts with County Features

In this step, we will merge the 9 new edge tag count columns into the main county-level feature dataframe. This way, each county will have its original features plus new features describing what kinds of Superfund sites are connected to it.

In [7]:
# Step 5: Merge the edge tag matrix with the original county feature dataframe
# Ensure FIPS codes are strings and padded for consistency
county_df['fips'] = county_df['fips'].astype(str).str.zfill(5)
edge_tag_matrix['fips'] = edge_tag_matrix['fips'].astype(str).str.zfill(5)

# Perform the merge on fips
enriched_county_df = pd.merge(county_df, edge_tag_matrix, on='fips', how='left')

# Fill missing values (i.e., counties with no Superfund sites) with 0
enriched_county_df.fillna(0, inplace=True)

# Preview the enriched dataset
enriched_county_df.head()


,county,state,fips,cancer_rate,avg_annual_count,five_year_trend,has_cancer_data,rural_rural,rural_urban,trend_,...,trend_stable,High_Medium,High_Old,High_Recent,Low_Medium,Low_Old,Low_Recent,Medium_Medium,Medium_Old,Medium_Recent
0,traverse county,minnesota,27155,693.5,37,2.2,1,True,False,False,...,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,polk county,texas,48373,679.5,436,-0.3,1,True,False,False,...,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,galax city,virginia,51640,655.0,55,1.7,1,True,False,False,...,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,greeley county,nebraska,31077,653.1,21,0.7,1,True,False,False,...,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,dewey county,south dakota,46041,634.5,28,1.3,1,True,False,False,...,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The 9 new columns (High_Recent, Medium_Old, etc.) have been added at the end.

Missing values for counties without Superfund sites were correctly filled with 0.0

Prepare the Feature Matrix for Model Training In this step, we will:

Drop non-numeric or non-feature columns (like county, state, and fips)

Keep only the columns you want to use as node features in the model

This cleaned-up matrix will be used as county_feature_matrix in GraphSAGE

In [9]:
# Step 6: Prepare the county feature matrix for modeling
# Drop non-numeric or identifier columns
columns_to_drop = ['county', 'state', 'fips']
county_feature_matrix = enriched_county_df.drop(columns=columns_to_drop)

# Ensure all remaining values are numeric and suitable for PyTorch
county_feature_matrix = county_feature_matrix.astype(float)

# Preview the final feature matrix
county_feature_matrix.head()


,cancer_rate,avg_annual_count,five_year_trend,has_cancer_data,rural_rural,rural_urban,trend_,trend_falling,trend_rising,trend_stable,High_Medium,High_Old,High_Recent,Low_Medium,Low_Old,Low_Recent,Medium_Medium,Medium_Old,Medium_Recent
0,693.5,37.0,2.2,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,679.5,436.0,-0.3,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,655.0,55.0,1.7,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,653.1,21.0,0.7,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,634.5,28.0,1.3,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Save the enriched county feature matrix to a CSV file
county_feature_matrix.to_csv("enriched_county_feature_matrix.csv", index=False)

# Confirm the file was saved
print("Saved as: enriched_county_feature_matrix.csv")


Saved as: enriched_county_feature_matrix.csv
